## Configuracion

Unica celda que cada miembro edita.

In [ ]:
NOMBRE          = "Camilo"  # opciones: Jesus, Camilo, Mateo, Sergio

# Ruta a la carpeta del corpus en Drive.
# Si ejecutas !ls /content/drive/MyDrive/ y no ves 'core',
# cambia esta variable con el nombre correcto de la carpeta en tu cuenta.
CARPETA_CORPUS  = "/content/drive/MyDrive/core"

REPO_RAW_URL    = "https://raw.githubusercontent.com/CamiloDS16/nlp-docs-cientificos-es/main/data/particiones.csv"
OUTPUT_PATH     = f"/content/drive/MyDrive/{NOMBRE.lower()}_corpus.parquet"
CHECKPOINT_PATH = f"/content/drive/MyDrive/{NOMBRE.lower()}_corpus_checkpoint.parquet"
CHECKPOINT_CADA = 5000

print(f"Miembro:  {NOMBRE}")
print(f"Corpus:   {CARPETA_CORPUS}")
print(f"Salida:   {OUTPUT_PATH}")

---
## Carga txt

Celdas originales del notebook de Jesus. Montan Drive, verifican la estructura
de carpetas y establecen la conexion con Google Drive API.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls /content/drive

In [ ]:
import os
carpeta = "/content/drive/MyDrive/core"

In [ ]:
!pip install --upgrade google-api-python-client

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
from googleapiclient.discovery import build

service = build('drive', 'v3')

In [ ]:
folder_id = "138g-tMrhRvE2-mjK4VaZlbRScawUNdII"

results = service.files().list(
    q=f"'{folder_id}' in parents and mimeType='text/plain'",
    pageSize=200,
    fields="files(id, name)"
).execute()

files = results.get("files", [])

len(files)

In [ ]:
from googleapiclient.http import MediaIoBaseDownload
import io

textos = []

for file in files:

    request = service.files().get_media(fileId=file['id'])

    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, request)

    done = False
    while not done:
        status, done = downloader.next_chunk()

    contenido = fh.getvalue().decode("utf-8", errors="ignore")
    textos.append(contenido)

print("Archivos cargados:", len(textos))

In [ ]:
print(textos[0][:2500])

---
## Carga completa filtrada por particion

Las celdas anteriores confirman que Drive esta montado y la conexion funciona.
A partir de aqui se carga el corpus completo asignado a cada miembro:
se filtra por `particiones.csv`, se leen los archivos directamente desde
la carpeta montada (mas rapido que la API para lecturas masivas), y se
guarda el resultado como parquet.

In [ ]:
!pip install -q pyarrow pandas tqdm

### Verificacion de acceso al folder

In [ ]:
# El corpus esta en un folder compartido accedido via acceso directo.
# En Colab, los accesos directos permiten detectar la carpeta (os.path.isdir = True)
# pero no leer archivos a traves del FUSE mount. Se usa la API directamente.
# El objeto 'service' y 'folder_id' ya fueron construidos por las celdas de Jesus.

print(f"Verificando acceso via API al folder: {folder_id}")
try:
    resultado_prueba = service.files().list(
        q=f"'{folder_id}' in parents and mimeType='text/plain' and trashed=false",
        pageSize=3,
        fields="files(id, name)"
    ).execute()
    archivos_prueba = resultado_prueba.get("files", [])
    if archivos_prueba:
        print(f"Acceso confirmado. Muestra de archivos en el folder:")
        for a in archivos_prueba:
            print(f"  {a['name']}")
    else:
        print("El folder es accesible pero no se encontraron archivos .txt.")
except Exception as e:
    print(f"Error al acceder al folder via API: {e}")
    raise

### Cargar particiones

In [ ]:
import pandas as pd

# leer asignaciones directamente desde el repo en GitHub
df_particiones = pd.read_csv(REPO_RAW_URL)

mis_docs = df_particiones[df_particiones["asignado_a"] == NOMBRE].copy()
mis_docs["doc_id"] = mis_docs["doc_id"].astype(str).str.zfill(7)

print(f"Total en particiones.csv:  {len(df_particiones):,}")
print(f"Asignados a {NOMBRE}:       {len(mis_docs):,}")

### Lookup y descarga via API

In [ ]:
from tqdm.notebook import tqdm
from googleapiclient.http import MediaIoBaseDownload
import io

# los doc_id en particiones.csv son indices posicionales (1 a 1.342.100),
# no nombres de archivo. el script de seleccion asigno el indice 1 al primer
# archivo ordenado alfabeticamente, el 2 al segundo, etc.
# hay que listar todos los archivos, ordenarlos, y seleccionar por posicion.

mis_indices = set(int(doc_id) for doc_id in mis_docs["doc_id"].tolist())

# paso 1: listar TODOS los archivos del folder y ordenarlos por nombre
print("Listando todos los archivos del folder para reconstruir el orden original...")
print("Esto puede tomar 10-20 minutos (1.342 paginas con pageSize=1000).")
print()

todos_los_archivos = []  # lista de (nombre, file_id)
page_token = None
pagina     = 0

while True:
    kwargs = dict(
        q=f"'{folder_id}' in parents and mimeType='text/plain' and trashed=false",
        pageSize=1000,
        fields="nextPageToken, files(id, name)",
    )
    if page_token:
        kwargs["pageToken"] = page_token

    resultados = service.files().list(**kwargs).execute()
    for archivo in resultados.get("files", []):
        todos_los_archivos.append((archivo["name"], archivo["id"]))

    pagina += 1
    if pagina % 100 == 0:
        print(f"  pagina {pagina} — archivos listados: {len(todos_los_archivos):,}")

    page_token = resultados.get("nextPageToken")
    if not page_token:
        break

# ordenar alfabeticamente por nombre para reproducir el orden del script original
todos_los_archivos.sort(key=lambda x: x[0])

print(f"Total archivos en folder: {len(todos_los_archivos):,}")
print()

# paso 2: seleccionar los archivos en las posiciones asignadas a este miembro
# los indices en particiones.csv son 1-based
lookup = {}  # indice -> (file_id, nombre)
for i, (nombre, file_id) in enumerate(todos_los_archivos, start=1):
    if i in mis_indices:
        lookup[i] = (file_id, nombre)

print(f"Archivos a descargar: {len(lookup):,} de {len(mis_indices):,} asignados")
if len(lookup) < len(mis_indices):
    print(f"  Advertencia: {len(mis_indices) - len(lookup):,} indices no tienen archivo correspondiente")
print()

# paso 3: descargar los archivos seleccionados
registros     = []
docs_fallidos = []

for i, (indice, (file_id, nombre)) in enumerate(tqdm(lookup.items(), desc=NOMBRE)):
    doc_id = f"{indice:07d}"
    try:
        request    = service.files().get_media(fileId=file_id)
        buffer     = io.BytesIO()
        downloader = MediaIoBaseDownload(buffer, request)
        hecho = False
        while not hecho:
            _, hecho = downloader.next_chunk()
        contenido = buffer.getvalue()
        try:
            texto = contenido.decode("utf-8")
        except UnicodeDecodeError:
            texto = contenido.decode("latin-1")
        registros.append({
            "doc_id":     doc_id,
            "filename":   nombre,
            "texto":      texto,
            "asignado_a": NOMBRE,
        })
    except Exception as e:
        docs_fallidos.append(doc_id)
        print(f"  error en indice {indice} ({nombre}): {e}")

    if (i + 1) % CHECKPOINT_CADA == 0 and registros:
        pd.DataFrame(registros).to_parquet(CHECKPOINT_PATH, index=False)
        print(f"  checkpoint: {i + 1} documentos procesados")

print(f"\nExitosos: {len(registros):,}")
print(f"Fallidos: {len(docs_fallidos):,}")

### Guardar parquet

In [ ]:
df_corpus = pd.DataFrame(registros)
df_corpus.to_parquet(OUTPUT_PATH, index=False)

tamano_mb = os.path.getsize(OUTPUT_PATH) / (1024 ** 2)

print(f"Ruta:      {OUTPUT_PATH}")
print(f"Docs:      {len(df_corpus):,}")
print(f"Fallidos:  {len(docs_fallidos):,}")
print(f"Tamano:    {tamano_mb:.1f} MB")

df_corpus.head(2)

### Analisis de calidad (opcional)

In [ ]:
# OPCIONAL
# El siguiente notebook fragmenta documentos en segmentos de 250-1000 palabras.
# Conviene saber cuantos documentos crudos tienen suficiente texto antes de continuar.

df_corpus["num_palabras"] = df_corpus["texto"].str.split().str.len()

print("Distribucion de longitud (palabras por documento):")
print(df_corpus["num_palabras"].describe().round(1).to_string())
print()

docs_ruido  = df_corpus[df_corpus["num_palabras"] < 100]
docs_utiles = df_corpus[df_corpus["num_palabras"] >= 250]

print(f"Docs < 100 palabras (ruido potencial):        {len(docs_ruido):,}  ({len(docs_ruido)/len(df_corpus)*100:.1f}%)")
print(f"Docs >= 250 palabras (aptos para fragmentar): {len(docs_utiles):,}  ({len(docs_utiles)/len(df_corpus)*100:.1f}%)")

### Registro en DVC

In [ ]:
# El parquet no va a GitHub directamente.
# Se copia a data/raw/ y se registra con DVC; solo el archivo .dvc se commitea al repo.

nombre_lower = NOMBRE.lower()

print("Despues de ejecutar este notebook, corre en tu terminal local:")
print()
print(f"  cp {OUTPUT_PATH} data/raw/{nombre_lower}_corpus.parquet")
print(f"  dvc add data/raw/{nombre_lower}_corpus.parquet")
print(f"  git add data/raw/{nombre_lower}_corpus.parquet.dvc .gitignore")
print(f"  git commit -m \"data: add {nombre_lower} partition\"")
print( "  git push")